<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Lab1_Claude_API_Tools.ipynb)

![M12 Lab1 Claude API Tools](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M12/assets/images/M12_Lab1_Claude_API_Tools_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab, set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    compare_responses,  # side-by-side Claude vs OpenAI view (used in section 5)
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,  # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,  # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL, # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M12 Lab 1, Claude API & Tools')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Make your <b>first call</b> to the Anthropic Claude API</li>
    <li>Understand the <b>differences</b> from OpenAI's chat API (system messages, message shape, content blocks)</li>
    <li>Use Claude's <b>tool use</b> feature to give the model a calculator and have it actually call it</li>
    <li>Stream a <b>multi-turn tool conversation</b> end to end</li>
  </ol>
</div>


## 🟧 Why a separate lab for Claude?

**Why this matters.** Every serious GenAI application ends up depending on more than one model provider. You may start on OpenAI, then add Claude because it is stronger at long documents, safer refusals, or a specific reasoning style, or because you want a fallback when one provider has an outage. The moment you reach for a second provider, you discover that the *ideas* transfer perfectly (chat, system prompts, tool calling) but the **API shape does not**. Field names change, the system prompt moves, and the response is structured differently. Those small differences are exactly where a first-time integration breaks.

**What we build here.** This lab makes you fluent in Anthropic's Python SDK so that "add Claude" becomes a ten minute change instead of a week of debugging. You have spent the course inside OpenAI's ecosystem; here you will make your first Claude call, see the three concrete differences from OpenAI's chat API, give Claude a real tool it can call, and finally run the same question through both providers side by side. By the end you will recognize the pattern that is shared across every provider and the surface details that are unique to Claude.

## 1️⃣ Setup, install the SDK and load your key

**Why this step exists.** Unlike the shared OpenAI utilities you have used all course, Claude needs its own client library and its own API key. The official package is `anthropic` on PyPI, and the key lives in a Colab Secret named `ANTHROPIC_API_KEY`, the exact same pattern you already use for `OPENAI_API_KEY`, just a different name.

**How the key is resolved.** In Colab, secrets are stored in the 🔑 sidebar and read at runtime with `userdata.get(...)`; we copy that value into the `ANTHROPIC_API_KEY` environment variable, which is where the Anthropic SDK looks by default. Doing it this way means the key never appears in the notebook text, so you can share the notebook without leaking your credentials. If the key is missing we stop immediately with a clear message rather than failing deep inside an API call later.

In [ ]:
# ==========================================================
# 1. Setup, install anthropic + load the key
# ==========================================================
import importlib.util, os

# Install the Anthropic SDK only if it is not already present in this runtime.
# find_spec returns None when the package is missing, so we install just once.
if importlib.util.find_spec("anthropic") is None:
    !pip install -q anthropic

# Pull the key from the same Colab secret pattern DADS uses everywhere.
try:
    from google.colab import userdata                     # only exists inside Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""  # copy secret -> env var
except (ImportError, ModuleNotFoundError):
    pass                                                   # not on Colab: rely on an existing env var

# Fail fast with a helpful message if the key never got set.
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise EnvironmentError("ANTHROPIC_API_KEY not found. In Colab: 🔑 sidebar → add a secret named ANTHROPIC_API_KEY.")

# Create the client. With no arguments it reads ANTHROPIC_API_KEY from the
# environment automatically, the same convenience setup_openai() gave us.
from anthropic import Anthropic
claude = Anthropic()
print("Anthropic client ready")                            # simple readiness check (setup, not a model result)

## 2️⃣ First call, a chat completion

**Why start here.** Before tools or anything fancy, you need to make one ordinary request and read the reply. This is where the three differences from OpenAI show up, and getting them right once means every later call just works.

Note three things compared to OpenAI:
1. The method is `claude.messages.create(...)`, not `chat.completions.create(...)`.
2. The **system prompt is a top-level argument** (`system=...`), not the first entry in the `messages` array. This is the difference that trips people up most often.
3. `max_tokens` is **required**, Anthropic treats it as part of the contract on every call, whereas OpenAI lets you omit it.

**Reading the reply.** A Claude response is not a single string. Its `.content` is a *list of content blocks*, and for a plain text answer the first block is the text. We will unpack this properly in the tool-use section; for now, `resp.content[0].text` gives you the words.

> **What is a token?** A *token* is the small chunk of text the model reads and writes in, very roughly a word or a piece of a word (about 4 characters of English). `max_tokens` caps how many tokens Claude may generate in its reply, and the `usage` numbers you print below report how many tokens went in and came out (which is what you are billed on).

In [ ]:
# ==========================================================
# 2. First call, a Claude chat completion
# ==========================================================
CLAUDE_MODEL = "claude-sonnet-4-6"   # a current Sonnet model: strong general-purpose default

resp = claude.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=300,                                  # REQUIRED by Anthropic on every call
    system="You are a precise technical writer. Answer in two sentences.", # system is a TOP-LEVEL arg, not a message
    messages=[
        {"role": "user", "content": "What is the difference between Anthropic's Sonnet and Opus tiers?"}
    ],
)

# Anthropic responses are a LIST of content blocks; for a plain text reply
# the first block is the text block, so .content[0].text is the answer.
text = resp.content[0].text
pretty_print(text, title="🟧 Claude reply", theme="yellow")     # show the model's words, nicely formatted

# usage reports the billed token counts for this call (input read + output written)
pp({
    "input tokens":  resp.usage.input_tokens,
    "output tokens": resp.usage.output_tokens,
}, title="Token usage")

## 3️⃣ Multi-turn, passing the history back

**Why this matters.** The API is **stateless**: Anthropic keeps no memory of your conversation between calls. If you want Claude to remember what was said two turns ago, *you* must resend the whole conversation every time. This is the same model OpenAI uses, so the habit transfers directly.

**How it works.** You keep a running `messages` list. After each reply you append the assistant's answer to that list, then append the next user question, and send the entire list again. Because the model sees the full transcript on every call, it can refer back to earlier turns as if it "remembered" them. The cost is that longer conversations send more tokens each turn, which is one reason production apps eventually trim or summarize old history. Watch below how turn 2 can answer a question that only makes sense given turn 1.

In [ ]:
# ==========================================================
# 3. Multi-turn, pass the history back each turn
# ==========================================================
# Start the running transcript with the first user message.
history = [
    {"role": "user", "content": "Hi! I'm preparing a graduate course on Generative AI."},
]

# Turn 1: send the history, get a reply.
r1 = claude.messages.create(model=CLAUDE_MODEL, max_tokens=200, system="You are a helpful TA.", messages=history)
history.append({"role": "assistant", "content": r1.content[0].text})   # append the assistant's turn to the transcript

# Turn 2: add the next user question, then resend the WHOLE history so Claude
# still "remembers" the course context from turn 1.
history.append({"role": "user", "content": "Suggest one good final-project topic for that course."})
r2 = claude.messages.create(model=CLAUDE_MODEL, max_tokens=200, system="You are a helpful TA.", messages=history)

pretty_print(r1.content[0].text, title="Turn 1", theme="yellow")   # first reply
pretty_print(r2.content[0].text, title="Turn 2", theme="yellow")   # second reply, grounded in turn 1

## 4️⃣ Tool use, give Claude a calculator

**Why tools exist.** A language model predicts text; it does not run code, so it cannot reliably do exact arithmetic, look up live data, or take real actions. **Tool use** (Anthropic's name; OpenAI calls it *function calling*) is the bridge: you describe a tool, and when a request needs it Claude asks *you* to run it and hand back the result. This is the exact same idea you saw in Module 3, only with Anthropic's field names.

**How the loop works.**
1. **You** describe a tool with a `name` and an `input_schema` (a small JSON Schema listing its arguments).
2. **The model** decides whether the request needs the tool. If yes, instead of answering in prose it returns a `tool_use` content block containing the arguments it extracted from the user's words.
3. **You** run the real tool with those arguments and send the output back as a `tool_result` block.
4. **The model** reads that result and writes its final answer.

We will wire a tiny calculator and watch Claude call it on a question that requires arithmetic.

**Three terms this uses.**
- **Content block**, a Claude reply is a *list* of typed pieces, not one string. A block is either `text` (words for the user) or `tool_use` (a request to run a tool).
- **`stop_reason`**, why the model stopped this turn. `"end_turn"` means it finished; `"tool_use"` means it paused to ask you to run a tool and hand back the result.
- **`tool_result`**, the block *you* send back carrying the tool's output, tagged with the `tool_use_id` so Claude knows which call it answers.

In [ ]:
# ==========================================================
# 4. Tool use, define the calculator tool
# ==========================================================
# TOOLS is the list of tool descriptions we pass to Claude. The model reads
# these descriptions (never our code) to decide when and how to call a tool.
TOOLS = [
    {
        "name": "calculator",                # the model refers to the tool by this exact name
        "description": (                       # a clear description is how the model knows WHEN to use it
            "Evaluate a basic arithmetic expression. Use ONLY for math the user asked about. "
            "Supports +, -, *, /, parentheses, and integer/decimal numbers."
        ),
        "input_schema": {                 # JSON Schema describing the arguments the model must fill in
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "An arithmetic expression to evaluate, e.g. '17.5 * 12'."}
            },
            "required": ["expression"],  # the model must always supply 'expression'
        },
    }
]

def run_calculator(expression: str) -> str:
    """Tiny safe-ish evaluator: only arithmetic characters are allowed."""
    import re
    # Reject anything that is not a digit, operator, parenthesis, dot, or space.
    # This whitelist blocks arbitrary code from sneaking into eval() below.
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "ERROR: only basic arithmetic characters allowed"
    try:
        # eval with empty builtins/globals so only arithmetic on the cleaned string can run.
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"               # return the error as text so Claude can react to it

In [ ]:
# ==========================================================
# 4. Tool use, run the tool-use loop
# ==========================================================
def chat_with_tools(user_message: str, max_iters: int = 4) -> str:
    """Run a tool-use loop until Claude returns a final assistant message."""
    msgs = [{"role": "user", "content": user_message}]     # start the conversation
    for step in range(max_iters):                          # cap the loop so it can never run forever
        resp = claude.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=400,
            tools=TOOLS,                                   # pass the tool list on EVERY call (stateless API)
            messages=msgs,
        )

        # stop_reason == "tool_use" means Claude paused to ask us to run a tool;
        # any other stop_reason means it finished, so we return its final text.
        if resp.stop_reason != "tool_use":
            # Find the first text block in the reply and return its words.
            return next((b.text for b in resp.content if b.type == "text"), "(no text)")

        # Otherwise: append the assistant turn (which contains the tool_use block)
        # verbatim, so Claude sees its own request on the next call.
        msgs.append({"role": "assistant", "content": resp.content})

        # Run every tool the model asked for and collect one tool_result per call.
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":                   # this block is a request to run a tool
                pp({"tool": block.name, "input": block.input}, title=f"Claude called a tool (step {step})")
                # Execute the real function; guard against an unknown tool name.
                output = run_calculator(**block.input) if block.name == "calculator" else f"unknown tool {block.name}"
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,               # ties this result to the exact call it answers
                    "content": output,
                })
        # Send all tool results back as a single user turn, then loop again.
        msgs.append({"role": "user", "content": tool_results})

    return "(loop limit reached)"                          # safety fallback if we never got a final answer

# A question that needs arithmetic: 14 modules x 3 recording hours x 5 = post hours.
answer = chat_with_tools(
    "I'm planning a course with 14 modules. If each module needs 3 hours of recording and post takes 5x, how many post hours total?"
)
pretty_print(answer, title="🧮 Final answer (with tool use)", theme="green")

## 5️⃣ Compare with OpenAI on the same question

**Why compare.** The best way to feel the difference between providers is to give them an identical task and read both answers next to each other. Same prompt, same word problem, but the two models may reach the answer differently, explain it differently, and (importantly) one may actually *call the tool* to compute while the other does the arithmetic in its head.

**How this is set up.** We reuse the `answer` Claude produced above (which went through the tool loop) and run the same question once through the OpenAI chat default. The `compare_responses(...)` helper renders them side by side so the contrast is easy to read. As you look at the two columns, notice not just whether the numbers match, but which explanation you would trust in production.

In [ ]:
# ==========================================================
# 5. Compare Claude with OpenAI on the same question
# ==========================================================
# Run the SAME word problem through the OpenAI chat default (client came from
# the shared setup_openai() in the API-check cell at the top of the lab).
openai_resp = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=[{
        "role": "user",
        "content": "I'm planning a course with 14 modules. If each module needs 3 hours of recording and post takes 5x, how many post hours total?"
    }],
)

# Show both answers side by side: Claude's (tool-assisted, from above) vs OpenAI's.
compare_responses({
    f"Claude · {CLAUDE_MODEL}":        answer,                                # tool-assisted answer from section 4
    f"OpenAI · {DEFAULT_CHAT_MODEL}":  openai_resp.choices[0].message.content, # OpenAI's plain answer
})

> **Pause and think: you vs the two models.** You just saw Claude and OpenAI answer the same word problem. Did they reach the same number? Which explanation was clearer, and which one actually *called the tool* to compute versus doing the arithmetic in its head? A model that computes with a tool is auditable; a model that does mental math can be confidently wrong. Jot your read below.

**Your notes** *(double-click to edit)*

- Same final answer? 
- Clearer explanation: 
- Which one I would ship, and why: 

## 🎯 Hands-on exercise

Now put the pieces together. Each task below reuses a mechanism from this lab, the tool loop, the model constant, and the multi-turn history pattern, so you cement how they fit.

1. **Add a `web_search`-style tool** (mocked, just return canned results) and wire it into `TOOLS` and the loop alongside `calculator`. Then ask Claude a question that needs *both* tools (for example, look something up and then do math on it) and watch it chain the two calls.
2. **Switch the model** to `claude-haiku-4-5` for the same calculator task. Haiku is smaller, faster, and cheaper, does it still call the tool correctly, or does it try to answer without it?
3. **Send a multi-turn tool conversation**, two arithmetic questions in a row through `chat_with_tools`. Inspect the `msgs` list afterward: what did Claude carry between turns, and where do the `tool_use` and `tool_result` blocks appear?

> 💡 The tool-use protocol is identical *conceptually* to OpenAI's function calling, only the field names change (`input_schema` vs `parameters`, `tool_use`/`tool_result` blocks vs `tool_calls`). Once you have internalized it for one provider, the other is a ten minute read of the docs.

In [ ]:
# Your turn.


---
*Next lab, the M12 Lab 2 self-study supplement covers Google's A2A protocol and OpenAI's Agents SDK. Same patterns, three platforms.*